## 01 - Data Generation & Validation for Polymer Formulations

This notebook documents the data-generation stage of a scientific ML workflow. We encode composition/process rules, synthesize physically plausible measurements, and prove that the resulting dataset can stand in for early laboratory hypotheses before any real experiments run.


In [ ]:
# Notebook bootstrap: asegura imports desde el root del proyecto
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_synthetic_formulations

sns.set_theme(style="whitegrid")

SEED = 7
N_FULL = 500

df = generate_synthetic_formulations(
    N_FULL,
    seed=SEED,
    noise_level=1.0,
    sum_tolerance=0.25,
)

df.head()

### 1) Constraint validation: composition sum (~100%)

The overall wt% sum must stay close to 100 to respect manufacturing bounds. We monitor the distribution of sums and enforce a tight tolerance when sampling.


In [ ]:
comp_cols = ["polymer_A", "polymer_B", "plasticizer", "filler", "stabilizer"]

composition_sum = df[comp_cols].sum(axis=1)
composition_sum.describe()

In [ ]:
plt.figure(figsize=(7, 3))

data_range = float(composition_sum.max() - composition_sum.min())
if data_range < 1e-6:
    # Si la suma es exacta (rango ~0), un histograma con muchos bins falla.
    plt.hist(composition_sum, bins=1, color="#4C78A8", edgecolor="white")
    plt.title("Composition sum (exact ~100 wt%)")
    plt.text(0.5, 0.9, "All sums are ~constant (good)", transform=plt.gca().transAxes, ha="center")
else:
    plt.hist(composition_sum, bins=30, color="#4C78A8", edgecolor="white")
    plt.title("Composition sum (should be ~100 wt%)")

plt.axvline(100.0, color="black", linewidth=1)
plt.xlabel("Sum")
plt.ylabel("Count")
plt.show()

**Interpretation**

- Stable composition sums show that the main formulation constraint is satisfied.
- This validation replaces the typical spreadsheet checks performed by process engineers.


### 2) Distributions and variability

An engineering workflow needs reasonable coverage of the design space. Inspect the histograms and violin plots to ensure every variable spans its realistic bounds without unrealistic spikes.


In [ ]:
cols_inputs = comp_cols + ["temperature", "curing_time", "pressure"]

df[cols_inputs].describe().T

In [ ]:
df[cols_inputs].hist(figsize=(12, 7), bins=25, color="#59A14F", edgecolor="white")
plt.suptitle("Input distributions", y=1.02)
plt.tight_layout()
plt.show()

### 3) Expected physical relationships (sanity checks)

We look for signals that match domain knowledge:

- **Filler vs. tensile strength:** positive up to a point, with diminishing returns.
- **Plasticizer vs. elongation:** higher plasticizer increases ductility but may soften the matrix.
- **Temperature:** bell-shaped response around the curing sweet spot.

These checks demonstrate that the synthetic data would pass a basic lab review.


In [ ]:
targets = ["tensile_strength", "elongation", "thermal_resistance"]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.ravel()

sns.scatterplot(ax=axes[0], data=df, x="filler", y="tensile_strength", alpha=0.6)
axes[0].set_title("filler vs tensile_strength")

sns.scatterplot(ax=axes[1], data=df, x="filler", y="elongation", alpha=0.6)
axes[1].set_title("filler vs elongation")

sns.scatterplot(ax=axes[2], data=df, x="plasticizer", y="elongation", alpha=0.6)
axes[2].set_title("plasticizer vs elongation")

sns.scatterplot(ax=axes[3], data=df, x="plasticizer", y="tensile_strength", alpha=0.6)
axes[3].set_title("plasticizer vs tensile_strength")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 4))
df[comp_cols].boxplot()
plt.title("Composition distribution check (outliers / plausibility)")
plt.ylabel("wt%")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="temperature", y="tensile_strength", alpha=0.6)
plt.title("temperature vs tensile_strength (non-linear optimum)")
plt.show()

### 4) Correlations (credibility)

Targets include non-linear effects and noise, so we do not expect perfect correlations. Instead, we verify that the pairwise structure reflects plausible interactions (slight negative filler-elongation, positive stabilizer-thermal resistance, etc.).


**Interpretation**

- Homogeneous coverage prevents the surrogate from relying on extrapolation only and helps quantify uncertainty.
- Engineers can trace each plot back to a physical rationale, which bolsters credibility when presenting the project.


In [ ]:
corr_cols = cols_inputs + targets
corr = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation heatmap (sanity check)")
plt.tight_layout()
plt.show()

### 5) Low-data scenario (~40 samples)

To mirror the reality of many labs, we also keep a down-sampled subset. It captures the same constraints but exposes how noisy the relationships become when only a few experiments exist.


In [ ]:
df_small = df.sample(40, random_state=42)

print("Full:", df.shape)
print("Small:", df_small.shape)

df_small.head()

**Interpretation**

- 
iller vs. 	ensile_strength: positive trend with diminishing returns at high loadings.
- plasticizer vs. elongation: positive slope that highlights the strength vs. ductility trade-off.
- 	emperature vs. 	hermal_resistance: mild optimum around the curing window.

These slices explain to reviewers why the surrogate models need to balance bias and variance.


In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df_small, x="filler", y="tensile_strength", alpha=0.8)
plt.title("Small dataset behavior (N=40): filler vs tensile_strength")
plt.show()

**Observations**

- With N = 40 the clouds are noisier and signals are less evident, matching real experimental campaigns.
- Maintaining both full and low-data views lets us demonstrate robustness when we hand off to modeling.


### Key Takeaways

- Composition constraints are respected, and the data reflects classical polymer trade-offs.
- The notebook doubles as a lab log: it records why the synthetic dataset is trustworthy enough to feed downstream ML and optimization.
